# SoftMeta Chatterbox TTS Server v0.9.2

Dependency-stable A100 40GB build for MOSS voice generation and long-form Avatar Talking.

- Select an **A100 GPU** before running the notebook.
- You may use **Run all**.
- Engine: `soft-meta/chatterbox-v2@v0.2.1`
- Server and UI: `soft-meta/Chatterbox-TTS-Server@v0.9.2`
- Generate Voice: official MOSS-TTS dependency set plus SpeechBrain 1.1.0
- Generate Video: Ditto PyTorch stable mode
- TensorRT is disabled by default because legacy TensorRT 8.6.1 does not match current Colab images.
- Long videos use checkpointed rendering on A100 40GB.

Start with a 10-20 second video test before a 10-30 minute render.


In [ ]:
%%bash
set -euo pipefail

apt-get update -qq
apt-get install -y -qq   ffmpeg libsndfile1 git git-lfs curl ca-certificates lsof   sox libsox-fmt-all build-essential

git lfs install

echo "GPU assigned by Colab:"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true

cd /content
rm -rf /content/bin
mkdir -p /content/bin

MICROMAMBA="/content/bin/micromamba"
MICROMAMBA_VERSION="2.6.2-1"
MICROMAMBA_URL="https://github.com/mamba-org/micromamba-releases/releases/download/${MICROMAMBA_VERSION}/micromamba-linux-64"

curl --fail --location --retry 5 --retry-delay 2 --retry-all-errors   --connect-timeout 30 "${MICROMAMBA_URL}" --output "${MICROMAMBA}"
chmod +x "${MICROMAMBA}"
"${MICROMAMBA}" --version

for ENV_NAME in sm311 moss312 avatar310; do
  if "${MICROMAMBA}" env list | awk '{print $1}' | grep -qx "${ENV_NAME}"; then
    "${MICROMAMBA}" env remove -n "${ENV_NAME}" -y || true
  fi
done

"${MICROMAMBA}" create -y -n sm311 -c conda-forge python=3.11 pip

echo "Main environment is ready. MOSS and Avatar installers create their own isolated environments."


In [ ]:
%%bash
set -euo pipefail

MM="/content/bin/micromamba"
cd /content
rm -rf chatterbox-v2 Chatterbox-TTS-Server MOSS-TTS

git clone --branch v0.2.1 --depth 1 https://github.com/soft-meta/chatterbox-v2.git
if ! git clone --branch v0.9.2 --depth 1 https://github.com/soft-meta/Chatterbox-TTS-Server.git; then
  echo "v0.9.2 tag is not published yet. Cloning v0.9.1 and applying the embedded v0.9.2 patch."
  rm -rf Chatterbox-TTS-Server
  git clone --branch v0.9.1 --depth 1 https://github.com/soft-meta/Chatterbox-TTS-Server.git
fi

python - <<'SOFTMETA_PATCH'
import base64
from pathlib import Path

project = Path("/content/Chatterbox-TTS-Server")
files = {"CHANGELOG.md":"IyBDaGFuZ2Vsb2cKCiMjIHYwLjkuMgoKLSBGaXhlZCB0aGUgZmF0YWwgU3BlZWNoQnJhaW4gaW1wb3J0IGNyYXNoIGNhdXNlZCBieSBhbiBpbmNvbXBhdGlibGUgVG9yY2hBdWRpbyBiYWNrZW5kIEFQSQotIFVwZGF0ZWQgdGhlIHNwZWFrZXIgY2hlY2tlciBmcm9tIFNwZWVjaEJyYWluIDEuMC4zIHRvIDEuMS4wCi0gUmVtb3ZlZCBjb25mbGljdGluZyBgc2NpcHlgLCBgdHJhbnNmb3JtZXJzYCwgYHRvcmNoYCBhbmQgYHRvcmNoYXVkaW9gIG92ZXJyaWRlcyBmcm9tIFNvZnRNZXRhIHZvaWNlIHJlcXVpcmVtZW50cwotIEFkZGVkIGEgZGVkaWNhdGVkIG9mZmljaWFsIE1PU1MgaW5zdGFsbGVyIHdpdGggYHBpcCBjaGVja2AgYW5kIGltcG9ydCB2ZXJpZmljYXRpb24KLSBBZGRlZCBhIHNhZmUgY29tcGF0aWJpbGl0eSBzaGltIGZvciBzdGFsZSBjYWNoZWQgU3BlZWNoQnJhaW4gd2hlZWxzCi0gQ2hhbmdlZCBDb2xhYiBBMTAwIDQwR0IgQXZhdGFyIFRhbGtpbmcgdG8gRGl0dG8gUHlUb3JjaCBzdGFibGUgbW9kZSBieSBkZWZhdWx0Ci0gU3RvcHBlZCBhdHRlbXB0aW5nIGxlZ2FjeSBUZW5zb3JSVCA4LjYuMSB1bmxlc3MgZXhwbGljaXRseSBlbmFibGVkCi0gRml4ZWQgYmFja2VuZCByZWFkaW5lc3Mgc28gVGVuc29yUlQgaXMgbmV2ZXIgc2VsZWN0ZWQgd2l0aG91dCBhbiBpbXBvcnRhYmxlIHJ1bnRpbWUKLSBBZGRlZCBlbnZpcm9ubWVudCBkaWFnbm9zdGljcyBhbmQgZGVwZW5kZW5jeSBjaGVja3MgYmVmb3JlIHNlcnZlciBzdGFydHVwCi0gVXBkYXRlZCBzZXJ2ZXIsIG5vdGVib29rIGFuZCBVSSBjYWNoZSBrZXlzIHRvIHYwLjkuMgoKIyMgdjAuOS4xCgotIEZpeGVkIHRoZSBDb2xhYiBmYWlsdXJlIHdoZW4gYHNjcmlwdHMvaW5zdGFsbF9kaXR0b19hMTAwLnNoYCB3YXMgYWJzZW50IGZyb20gdGhlIEdpdEh1YiB0YWcKLSBBZGRlZCBhbiBlbWJlZGRlZCBpbnN0YWxsZXIgZmFsbGJhY2sgaW5zaWRlIHRoZSBub3RlYm9vawotIEFkZGVkIEExMDAgNDBHQiBWUkFNIGRldGVjdGlvbiBhbmQgc2FmZXIgd29ya2luZyByZXNvbHV0aW9ucwotIE1hZGUgY2hlY2twb2ludGVkIHJlbmRlcmluZyB3aXRoIHR3by1taW51dGUgc2VjdGlvbnMgdGhlIGxvbmctdmlkZW8gZGVmYXVsdAotIEF1dG9tYXRpY2FsbHkgcHJvdGVjdHMgY29udGludW91cyBqb2JzIGxvbmdlciB0aGFuIGZpdmUgbWludXRlcyBvbiA0MEdCIEdQVXMKLSBBZGRlZCBHUFUgcHJvZmlsZSwgZWZmZWN0aXZlIHJlbmRlciBtb2RlIGFuZCBnZW5lcmF0aW9uIHNpemUgdG8gdmlkZW8gcmVzdWx0cwotIFVwZGF0ZWQgc2VydmVyLCBub3RlYm9vayBhbmQgVUkgY2FjaGUga2V5cyB0byB2MC45LjEKCiMjIHYwLjkuMAoKLSBBZGRlZCBhIEdlbmVyYXRlIFZpZGVvIHRhYiB0aGF0IGFsd2F5cyBmb2xsb3dzIHRoZSBsYXN0IGN1cnJlbnQgQXVkaW8gd29ya3NwYWNlCi0gQWRkZWQgYXZhdGFyIGltYWdlIGFuZCBzZXBhcmF0ZSBhdWRpbyB1cGxvYWQgc3RvcmFnZQotIEFkZGVkIGRpcmVjdCBjb21wbGV0ZWQgQXVkaW8gMeKAkzUgc2VsZWN0aW9uIGZvciB2aWRlbyBjcmVhdGlvbgotIEFkZGVkIGFuIGlzb2xhdGVkIEExMDAgRGl0dG8gVGVuc29yUlQvUHlUb3JjaCB3b3JrZXIKLSBBZGRlZCBjb250aW51b3VzIGFuZCBzaWxlbmNlLWF3YXJlIGNoZWNrcG9pbnRlZCBsb25nLXZpZGVvIHJlbmRlcmluZwotIEFkZGVkIGEgcGVyc2lzdGVudCB2aWRlbyBxdWV1ZSB3aXRoIHByb2dyZXNzLCBFVEEsIGNhbmNlbGxhdGlvbiBhbmQgbG9ncwotIEFkZGVkIHBvcnRyYWl0LCBsYW5kc2NhcGUgYW5kIHNxdWFyZSB2aWRlbyBkZWxpdmVyeSBzZXR0aW5ncwotIEFkZGVkIGZpbmFsIEguMjY0L0FBQyBlbmNvZGluZyB3aXRoIG9yaWdpbmFsLWF1ZGlvIHJlc3RvcmF0aW9uCi0gQWRkZWQgZHVyYXRpb24tZHJpZnQgYW5kIGxvbmctZnJlZXplIHRlY2huaWNhbCBxdWFsaXR5IGNoZWNrcwotIEFkZGVkIEF2YXRhciBUYWxraW5nIEFQSSBlbmRwb2ludHMsIGRvY3VtZW50YXRpb24gYW5kIENvbGFiIGluc3RhbGxlcgotIFVwZGF0ZWQgc2VydmVyLCBVSSBjYWNoZSBrZXlzIGFuZCByZWxlYXNlIHJlZmVyZW5jZXMgdG8gdjAuOS4wCgojIyB2MC44LjAKCi0gUmVwbGFjZWQgUXdlbjMtVFRTIFZvaWNlRGVzaWduIHdpdGggb2ZmaWNpYWwgTU9TUyBWb2ljZUdlbmVyYXRvciBhcyB0aGUgcHJpbWFyeSBHZW5lcmF0ZSBWb2ljZSBlbmdpbmUKLSBBZGRlZCBhIHBpbm5lZCBhbmQgdmFsaWRhdGVkIE1PU1MgbW9kZWwgc25hcHNob3Qgd2l0aCBhbiBpc29sYXRlZCBQeXRob24gMy4xMiBDb2xhYiBlbnZpcm9ubWVudAotIFJlYnVpbHQgdm9pY2UgaW5zdHJ1Y3Rpb25zIGFyb3VuZCBjb21wYWN0IGlkZW50aXR5LWZpcnN0IE1PU1MgcHJvbXB0cwotIEFkZGVkIGJyb2FkZXIgQW1lcmljYW4gc3BlZWNoIGJhY2tncm91bmRzIGFuZCBwcm90ZWN0ZWQgbG93LCBtZWRpdW0gYW5kIGhpZ2ggcGl0Y2ggZGl2ZXJzaXR5IGF0IGV2ZXJ5IGFnZQotIEFkZGVkIGV4cGxpY2l0IGd1aWRhbmNlIGFnYWluc3QgZ2xvYmFsIHNsb3dkb3duLCBzdHJldGNoZWQgdm93ZWxzIGFuZCBmaXhlZCBzdG9wLXN0YXJ0IHBhdXNlcwotIEFkZGVkIGFjb3VzdGljIHF1YWxpdHkgc2NyZWVuaW5nIGZvciBkZWFkIGFpciwgY2xpcHBpbmcsIGFjdGl2ZSBzcGVlY2ggbGV2ZWwsIGR5bmFtaWNzIGFuZCBhZ2UtYXdhcmUgcGF1c2UgYmVoYXZpb3VyCi0gQWRkZWQgTmF0dXJhbG5lc3Mgc2NvcmUgYW5kIGFjb3VzdGljIGRpYWdub3N0aWNzIHRvIGNhbmRpZGF0ZSBjYXJkcwotIEFkZGVkIHNlcGFyYXRlIGR1cGxpY2F0ZS1pZGVudGl0eSBhbmQgbG93LXF1YWxpdHkgcmVqZWN0aW9uIGNvdW50ZXJzCi0gUHJlc2VydmVkIHVwIHRvIDEyIGludGVybmFsIGF0dGVtcHRzLCBFQ0FQQSBwYWlyd2lzZSBjb21wYXJpc29uIGFuZCBhdXRvbWF0aWMgZHVwbGljYXRlIHJlamVjdGlvbgotIEtlcHQgQ2hhdHRlcmJveCBhcyB0aGUgbG9uZy1mb3JtIGNsb25pbmcgZW5naW5lIGZvciBzZWxlY3RlZCBnZW5lcmF0ZWQgdm9pY2VzCi0gVXBkYXRlZCBzZXJ2ZXIsIFVJIGNhY2hlIGtleXMsIERvY2tlciBhbmQgQ29sYWIgcmVsZWFzZSByZWZlcmVuY2VzIHRvIHYwLjguMAoKIyMgdjAuNy4wCgotIEFkZGVkIGlkZW50aXR5LWZpcnN0IHNwZWFrZXIgcHJvbXB0aW5nIHNvIGEgbmV3IGh1bWFuIGlkZW50aXR5IGlzIGVzdGFibGlzaGVkIGJlZm9yZSBhZ2UgYW5kIGVtb3Rpb24KLSBBZGRlZCBzdGFibGUgaWRlbnRpdHkgY29kZXMgYW5kIGV4cGFuZGVkIHZvY2FsIGFuYXRvbXksIHNwZWN0cmFsIGNvbG91ciwgbmFzYWxpdHksIHZvd2VsLCBjb25zb25hbnQgYW5kIHNwZWFraW5nLWhhYml0IGRpbWVuc2lvbnMKLSBBZGRlZCBzdHJpY3Qgb3Zlci1nZW5lcmF0aW9uOiB1cCB0byAxMiBhdHRlbXB0cyBhcmUgc2VhcmNoZWQgdG8gZmlsbCB0aGUgcmVxdWVzdGVkIDHigJM0IGNhbmRpZGF0ZSBzbG90cwotIEFkZGVkIGltbWVkaWF0ZSBTcGVlY2hCcmFpbiBjb21wYXJpc29uIGFnYWluc3Qgc2F2ZWQgdm9pY2VzIGFuZCBldmVyeSBlYXJsaWVyIGF0dGVtcHQgaW4gdGhlIGN1cnJlbnQgYmF0Y2gKLSBBZGRlZCBhdXRvbWF0aWMgcmVqZWN0aW9uIG9mIGNhbmRpZGF0ZXMgYWJvdmUgdGhlIHNwZWFrZXItc2ltaWxhcml0eSB0aHJlc2hvbGQKLSBBZGRlZCByZXZpZXcgZmFsbGJhY2sgb25seSB3aGVuIHRoZSBzdHJpY3Qgc2VhcmNoIGNhbm5vdCBmaWxsIGFsbCByZXF1ZXN0ZWQgc2xvdHMKLSBSZXBsYWNlZCBmaXJzdC1jYW5kaWRhdGUg4oCcMTAwJSBkaWZmZXJlbnTigJ0gb3V0cHV0IHdpdGggYSB0cnV0aGZ1bCBiYXNlbGluZSBzdGF0dXMKLSBDaGFuZ2VkIHRoZSBVSSB0byBzaG93IGNsb3Nlc3Qgc3BlYWtlciBzaW1pbGFyaXR5IGluc3RlYWQgb2YgYSBtaXNsZWFkaW5nIGRpZmZlcmVuY2UgcGVyY2VudGFnZQotIEFkZGVkIGNvbXBhcmlzb24gY291bnQsIGlkZW50aXR5IGNvZGUgYW5kIHJpY2hlciBpZGVudGl0eSB0cmFpdHMgdG8gZXZlcnkgY2FuZGlkYXRlIGNhcmQKLSBSYWlzZWQgdGhlIGRlZmF1bHQgcmVwZWF0ZWQtaWRlbnRpdHkgdGhyZXNob2xkIGZyb20gMC42OCB0byAwLjcyCi0gVXBkYXRlZCBzZXJ2ZXIsIFVJIGNhY2hlIGtleXMgYW5kIENvbGFiIHJlbGVhc2UgcmVmZXJlbmNlcyB0byB2MC43LjAKCiMjIHYwLjYuMAoKLSBGaXhlZCBRd2VuIGNhbmRpZGF0ZSBwcmV2aWV3IHBsYXllcnMgYmVpbmcgcmVidWlsdCBhbmQgc3RvcHBlZCBieSBiYWNrZ3JvdW5kIHF1ZXVlIHBvbGxpbmcKLSBBZGRlZCBvbmUtYXQtYS10aW1lIGNhbmRpZGF0ZSBwbGF5YmFjayB3aXRob3V0IHJlZnJlc2hpbmcgdGhlIG90aGVyIHByZXZpZXcgcGxheWVycwotIEFkZGVkIHN0cm9uZ2VyIGlkZW50aXR5IGZhbWlsaWVzLCBhZ2UtY29uZGl0aW9uZWQgdm9jYWwgY2hhcmFjdGVyIGFuZCB2YXJpZWQgUXdlbiBzYW1wbGluZyBwcm9maWxlcwotIFRpZ2h0ZW5lZCBnZW5lcmF0ZWQtdm9pY2Ugc2ltaWxhcml0eSBzY3JlZW5pbmcgYW5kIGJsb2NrZWQgc2F2aW5nIGNhbmRpZGF0ZXMgbWFya2VkIHRvbyBzaW1pbGFyCi0gQXV0by1hZHZhbmNlZCB0aGUgdmFyaWF0aW9uIHNlZWQgYWZ0ZXIgZWFjaCBnZW5lcmF0ZWQgYmF0Y2gKLSBDaGFuZ2VkIHRoZSBkZWZhdWx0IHdvcmtzcGFjZSBmcm9tIEF1ZGlvIDEgKyBBdWRpbyAyIHRvIEF1ZGlvIDEgb25seQotIEFkZGVkIHJlbW92YWJsZSBtaW51cyBjb250cm9scyB0byBBdWRpbyAyIHRocm91Z2ggQXVkaW8gNQotIEFkZGVkIGEgc2Vjb25kIEFQSSBsaW5rIGxhYmVsbGVkIENoYXR0ZXJib3ggVFRTIEFQSQotIFJlZmluZWQgdGhlIGludGVyZmFjZSB3aXRoIGEgcHJvZmVzc2lvbmFsIGhlYWRlciwgdmlzdWFsIGhpZXJhcmNoeSwgc3BhY2luZywgZm9ybSBjb250cm9scyBhbmQgY2FuZGlkYXRlIGNhcmRzCi0gQWRkZWQgZml2ZSBidW5kbGVkIHVzZXItcHJvdmlkZWQgcHJlZGVmaW5lZCBBbWVyaWNhbiBtYWxlIHJlZmVyZW5jZSB2b2ljZXMKLSBBZGRlZCBwcmVkZWZpbmVkIHZvaWNlIGltcG9ydGluZyBkaXJlY3RseSBmcm9tIHRoZSBQcmVkZWZpbmVkIFZvaWNlcyB0YWIKLSBVcGRhdGVkIHNlcnZlciBhbmQgVUkgdmVyc2lvbiB0byAwLjYuMAoKCiMjIHYwLjUuMgoKLSBQaW5uZWQgYSBjb21wbGV0ZSBvZmZpY2lhbCBRd2VuMy1UVFMgVm9pY2VEZXNpZ24gbW9kZWwgcmV2aXNpb24KLSBBZGRlZCBkZWRpY2F0ZWQgbG9jYWwgc25hcHNob3QgZG93bmxvYWQgYW5kIHJlcXVpcmVkLWZpbGUgdmFsaWRhdGlvbgotIEZpeGVkIHN0YWxlIHNwZWVjaC10b2tlbml6ZXIgZmVhdHVyZSBleHRyYWN0b3IgY2FjaGUgZmFpbHVyZXMKLSBJbnN0YWxsZWQgU29YIGluIHRoZSBDb2xhYiBlbnZpcm9ubWVudAotIEZpeGVkIHRoZSBjdXJyZW50IHNlcnZlci1sb2cgcGF0aAoKCiMjIHYwLjUuMQoKLSBGaXhlZCBhbiB1bnRlcm1pbmF0ZWQgUHl0aG9uIHN0cmluZyBpbiB0aGUgUXdlbjMtVFRTIENvbGFiIGVudmlyb25tZW50IHZlcmlmaWNhdGlvbiBjZWxsLgotIEFkZGVkIGEgY2xlYW4gdmVyaWZpY2F0aW9uIGhlYWRpbmcgdXNpbmcgdHdvIHZhbGlkIGBwcmludCgpYCBjYWxscy4KLSBVcGRhdGVkIENvbGFiIHJlbGVhc2UgcmVmZXJlbmNlcyBhbmQgY2FjaGUtYnVzdGluZyBVSSB2ZXJzaW9uIHN0cmluZ3MuCi0gTm8gY2hhbmdlcyB0byB2b2ljZSBkZXNpZ24sIHF1ZXVlIHByb2Nlc3NpbmcsIG9yIENoYXR0ZXJib3ggZ2VuZXJhdGlvbi4KCiMjIHYwLjUuMAoKLSBSZXBsYWNlZCBQYXJsZXItVFRTIEdlbmVyYXRlIFZvaWNlIHdpdGggb2ZmaWNpYWwgUXdlbjMtVFRTIFZvaWNlRGVzaWduCi0gQWRkZWQgMuKAkzQgZmljdGlvbmFsIHZvaWNlIGNhbmRpZGF0ZXMgcGVyIHJlcXVlc3QKLSBBZGRlZCBhZ2UtYXdhcmUgbmF0dXJhbCBwaHJhc2UgYW5kIHBhdXNlIGluc3RydWN0aW9ucyB3aXRob3V0IGdsb2JhbCBzbG93ZG93bgotIEFkZGVkIHNlZWQtZHJpdmVuIGlkZW50aXR5IHZhcmlhdGlvbiBhY3Jvc3MgcGl0Y2gsIHJlc29uYW5jZSwgdGV4dHVyZSwgYXJ0aWN1bGF0aW9uLCBwZXJzb25hbGl0eSBhbmQgY2FkZW5jZQotIEFkZGVkIGNhbmRpZGF0ZSBwcmV2aWV3LCBkb3dubG9hZCwgc2F2ZSBhbmQgcmV1c2Ugd29ya2Zsb3cKLSBBZGRlZCBvcHRpb25hbCBTcGVlY2hCcmFpbiBFQ0FQQSB2b2ljZS1kaWZmZXJlbmNlIGNoZWNraW5nIGFuZCBlbWJlZGRpbmcgY2FjaGUKLSBBZGRlZCBpc29sYXRlZCBRd2VuMy1UVFMgQ29sYWIgZW52aXJvbm1lbnQKLSBVcGRhdGVkIHNlcnZlciBhbmQgVUkgdmVyc2lvbiB0byAwLjUuMAoKIyMgdjAuNC4wCgotIEFkZGVkIGV4cGxpY2l0IHNwZWFrZXIgYWdlLCBnZW5kZXIsIFVTIEVuZ2xpc2ggYWNjZW50IGFuZCBlbW90aW9uIGZpZWxkcwotIEFkZGVkIGFnZS1iYXNlZCBwYWNpbmcgZnJvbSBtYXR1cmUgYWR1bHQgdGhyb3VnaCA5MCsgZWxkZXJseSBkZWxpdmVyeQotIEFkZGVkIGEgTmF0dXJhbCBIdW1hbiBWb2ljZSBGb3JtdWxhIGFuZCBVSSBwcm9maWxlIHByZXZpZXcKLSBBZGRlZCBzdGFibGUgZ2VuZGVyLW1hdGNoZWQgUGFybGVyIHNwZWFrZXIgaWRlbnRpdHkgc2VsZWN0aW9uIGJ5IHNlZWQKLSBBZGRlZCBnZW50bGUgcGl0Y2gtcHJlc2VydmluZyBhZ2UgdGVtcG8gY29ycmVjdGlvbgotIEFkZGVkIGF1dG9tYXRpYyByZWNvbW1lbmRlZCBmaW5hbCBDaGF0dGVyYm94IHNwZWVkCi0gQWRkZWQgZ2VuZXJhdGVkIHZvaWNlIEpTT04gcHJvZmlsZSBtZXRhZGF0YQotIFVwZGF0ZWQgc2VydmVyIGFuZCBVSSB2ZXJzaW9uIHRvIDAuNC4wCgojIyB2MC4zLjEKCi0gRml4ZWQgR2VuZXJhdGUgVm9pY2UgaW1wb3J0IGZhaWx1cmVzIGNhdXNlZCBieSBpbmNvbXBhdGlibGUgVHJhbnNmb3JtZXJzIHJlcXVpcmVtZW50cwotIEFkZGVkIGFuIGlzb2xhdGVkIFBhcmxlci1UVFMgdmlydHVhbCBlbnZpcm9ubWVudCB0aGF0IHNoYXJlcyB0aGUgbWFpbiBDVURBIFB5VG9yY2ggaW5zdGFsbGF0aW9uCi0gQWRkZWQgYSBkZWRpY2F0ZWQgdm9pY2Ugd29ya2VyIHByb2Nlc3Mgd2l0aCBkZXRhaWxlZCBkaWFnbm9zdGljcyBhbmQgdGltZW91dCBoYW5kbGluZwotIEtlcHQgQ2hhdHRlcmJveCBhbmQgUGFybGVyLVRUUyBkZXBlbmRlbmNpZXMgc2VwYXJhdGVkIHdpdGhvdXQgZHVwbGljYXRpbmcgdGhlIEdQVSBqb2IgcXVldWUKLSBVcGRhdGVkIENvbGFiIGFuZCBEb2NrZXIgaW5zdGFsbGF0aW9uIGZsb3dzCgojIyB2MC4zLjAKCi0gUmVwbGFjZWQgdGhlIGR1cGxpY2F0ZSB2aXNpYmxlIGJyb3dzZXIgYXVkaW8gcGxheWVyIHdpdGggYSBoaWRkZW4gYXVkaW8gZW5naW5lCiAgYW5kIGN1c3RvbSBwbGF5YmFjayBjb250cm9scyBkaXJlY3RseSBiZWxvdyB0aGUgbWFpbiB3YXZlZm9ybQotIEFkZGVkIGEgbGl2ZSB3YXZlZm9ybSBwbGF5aGVhZCwgY3VycmVudC90b3RhbCB0aW1lIGFuZCBzZWVrIHNsaWRlcgotIFJlbW92ZWQgZml4ZWQg4oCcS2VlcCBmaXJzdOKAnSBxdWljay10aW1lIGJ1dHRvbnMgZnJvbSBDdXQgR2VuZXJhdGVkIEF1ZGlvCi0gQWRkZWQgYSBsaW5rZWQgU29mdE1ldGEgQ2hhdHRlcmJveCBUVFMgZm9vdGVyIGNyZWRpdAotIEluY3JlYXNlZCB0eXBvZ3JhcGh5IHRocm91Z2hvdXQgdGhlIHN0dWRpbywgcXVldWUgbW9uaXRvciBhbmQgZWRpdG9yCi0gQWRkZWQgUmVtb3ZlIEFsbCBmb3IgY29tcGxldGVkIGpvYnMsIGdlbmVyYXRlZCBvdXRwdXQgZmlsZXMsIHRpdGxlcyBhbmQgc2NyaXB0cwotIEFkZGVkIHJlbW92YWJsZSBBdWRpbyAz4oCTNSB0YWJzIHdpdGggYSBtaW51cyBidXR0b24KLSBSZW1vdmVkIE1vZGVsIERlZmF1bHQgZnJvbSB0aGUgdmlzaWJsZSB2b2ljZS1tb2RlIGNob2ljZXMKLSBBZGRlZCBHZW5lcmF0ZSBWb2ljZSB3aXRoIHNwZWFrZXIgZGVzY3JpcHRpb24sIHNhbXBsZSB0ZXh0LCBzZWVkLCBwcmV2aWV3LAogIGRvd25sb2FkIGFuZCByZXVzYWJsZSBzYXZlZCB2b2ljZSByZWZlcmVuY2VzCi0gQWRkZWQgbGF6eSBQYXJsZXItVFRTIE1pbmkgdjEuMSBpbnRlZ3JhdGlvbiBmb3IgdGV4dC1kZXNjcmliZWQgcmVmZXJlbmNlIFdBVnMKLSBBZGRlZCBnZW5lcmF0ZWQgdm9pY2Ugc3RvcmFnZSBhbmQgQ2hhdHRlcmJveCBjbG9uaW5nIHN1cHBvcnQKLSBVcGRhdGVkIHRoZSBzZXJ2ZXIgdG8gYDAuMy4wYCBhbmQga2VwdCB0aGUgZW5naW5lIHBpbm5lZCB0byBgdjAuMi4xYAoKIyMgdjAuMi4xCgotIFBpbm5lZCBgc2V0dXB0b29sczw4MWAgZm9yIGNvbXBhdGliaWxpdHkgd2l0aCB0aGUgY3VycmVudCBvZmZpY2lhbCBQZXJUaCBwYWNrYWdlCi0gVmVyaWZpZWQgdGhhdCBgUGVydGhJbXBsaWNpdFdhdGVybWFya2VyYCBpcyBjYWxsYWJsZSBiZWZvcmUgbGF1bmNoaW5nIHRoZSBzZXJ2ZXIKLSBQcmVzZXJ2ZWQgdGhlIHNhZmUgQ29sYWIgbGF1bmNoZXIgdGhhdCBkb2VzIG5vdCBzdG9wIHRoZSBzZXJ2ZXIgZHVyaW5nIFJ1biBhbGwKLSBTaG93ZWQgdGhlIHJlYWwgc3RhcnR1cCBlcnJvciBpbnNpZGUgQ29sYWIgYmVmb3JlIG9wZW5pbmcgdGhlIHByb3h5IFVSTAoKIyMgdjAuMi4wCgotIFJlYnVpbHQgdGhlIGJyb3dzZXIgc3R1ZGlvIHRvIG1hdGNoIHRoZSBBemFkIG11bHRpLWF1ZGlvIHdvcmtmbG93Ci0gQWRkZWQgc2VydmVyLXNpZGUgc2VxdWVudGlhbCBBdWRpbyAx4oCTNSBxdWV1ZQotIEFkZGVkIGxpdmUgd29yZCBwcm9ncmVzcywgRVRBIGFuZCBhdWRpby1sZW5ndGggZXN0aW1hdGVzCi0gQWRkZWQgY29tcGxldGVkLWF1ZGlvIHByZXZpZXcgaW4gUXVldWUgTW9uaXRvcgotIEFkZGVkIHNlcnZlciB3YXZlZm9ybSBwZWFrcywgem9vbSwgcGFuLCBtb3VzZSB0aW1lIGFuZCBTdGFydC9FbmQgc2VsZWN0aW9uCi0gQWRkZWQgc2VsZWN0ZWQsIFBhcnQgT25lIGFuZCBQYXJ0IFR3byBkb3dubG9hZHMgd2l0aCB0aXRsZS1iYXNlZCBmaWxlbmFtZXMKCiMjIHYwLjEuMAoKLSBJbml0aWFsIHByb3RvdHlwZQo=","GITHUB_UPDATE_V0.9.2.md":"IyBTb2Z0TWV0YSB2MC45LjIgR2l0SHViIHVwZGF0ZQoKVXBkYXRlIGBzb2Z0LW1ldGEvQ2hhdHRlcmJveC1UVFMtU2VydmVyYCBhbmQga2VlcApgc29mdC1tZXRhL2NoYXR0ZXJib3gtdjJAdjAuMi4xYC4KCioqQ29tbWl0OioqCgpgRml4IE1PU1MgZGVwZW5kZW5jaWVzIGFuZCBBMTAwIEF2YXRhciBydW50aW1lIHYwLjkuMmAKCioqVGFnOioqCgpgdjAuOS4yYAoKKipSZWxlYXNlIHRpdGxlOioqCgpgU29mdE1ldGEgQ2hhdHRlcmJveCBUVFMgU2VydmVyIHYwLjkuMmAKCioqUmVsZWFzZSBub3RlOioqCgpgRml4ZWQgdGhlIFNwZWVjaEJyYWluIGFuZCBUb3JjaEF1ZGlvIGltcG9ydCBjcmFzaCwgcmVtb3ZlZCBjb25mbGljdGluZyBNT1NTIGRlcGVuZGVuY3kgcGlucywgYWRkZWQgdmVyaWZpZWQgaXNvbGF0ZWQgaW5zdGFsbGVycywgY2hhbmdlZCBDb2xhYiBBMTAwIDQwR0IgQXZhdGFyIFRhbGtpbmcgdG8gc3RhYmxlIERpdHRvIFB5VG9yY2ggbW9kZSwgYW5kIG1hZGUgVGVuc29yUlQgb3B0aW9uYWwgaW5zdGVhZCBvZiBmYWlsaW5nIGR1cmluZyBpbnN0YWxsYXRpb24uYAoKVXNlIGBSRUxFQVNFX05PVEVTX1YwLjkuMi5tZGAgYXMgdGhlIGZ1bGwgR2l0SHViIHJlbGVhc2UgZGVzY3JpcHRpb24uCg==","README.md":"IyBTb2Z0TWV0YSBDaGF0dGVyYm94IFRUUyBTZXJ2ZXIKCkEgc2VsZi1ob3N0ZWQgc3BlZWNoIGFuZCBsb25nLWZvcm0gYXZhdGFyIHN0dWRpbyBtYWludGFpbmVkIGJ5ICoqU29mdE1ldGEqKi4KVGhlIHNlcnZlciwgcGVyc2lzdGVudCBxdWV1ZXMsIGJyb3dzZXIgVUksIHdhdmVmb3JtIGVkaXRvciwgdm9pY2UtY2FuZGlkYXRlCndvcmtmbG93IGFuZCBhdmF0YXIgb3JjaGVzdHJhdGlvbiBhcmUgU29mdE1ldGEgY29kZS4KCiMjIHYwLjkuMjogZGVwZW5kZW5jeS1zdGFibGUgQTEwMCA0MEdCIGJ1aWxkCgotIFJlbW92ZXMgdGhlIGluY29tcGF0aWJsZSBTY2lQeSBvdmVycmlkZSBmcm9tIHRoZSBpc29sYXRlZCBNT1NTIGVudmlyb25tZW50LgotIFVzZXMgdGhlIG9mZmljaWFsIE1PU1MtVFRTIGB0b3JjaC1ydW50aW1lYCBkZXBlbmRlbmN5IHNldC4KLSBVcGRhdGVzIHRoZSBzcGVha2VyIGNoZWNrZXIgdG8gU3BlZWNoQnJhaW4gMS4xLjAgYW5kIFNvdW5kRmlsZSBhdWRpbyBJL08uCi0gQWRkcyBhIGNvbXBhdGliaWxpdHkgZ3VhcmQgZm9yIHN0YWxlIGNhY2hlZCBTcGVlY2hCcmFpbiB3aGVlbHMuCi0gVXNlcyBEaXR0byBQeVRvcmNoIGFzIHRoZSBzdGFibGUgQ29sYWIgQTEwMCA0MEdCIGF2YXRhciBiYWNrZW5kLgotIFNraXBzIGxlZ2FjeSBUZW5zb3JSVCA4LjYuMSBieSBkZWZhdWx0IGluc3RlYWQgb2YgcHJpbnRpbmcgYSBmYWlsZWQgYnVpbGQuCi0gVmVyaWZpZXMgZXZlcnkgaXNvbGF0ZWQgZW52aXJvbm1lbnQgd2l0aCBgcGlwIGNoZWNrYCBiZWZvcmUgc3RhcnRpbmcgdGhlIHNlcnZlci4KLSBLZWVwcyBjaGVja3BvaW50ZWQgMTAtMzAgbWludXRlIHZpZGVvIHJlbmRlcmluZyBhbmQgcmVkdWNlZCB3b3JraW5nIHJlc29sdXRpb24uCgoKIyMgdjAuOC4wOiBNT1NTIHVuaXF1ZSB2b2ljZSBnZW5lcmF0aW9uCgpUaGUgKipHZW5lcmF0ZSBWb2ljZSoqIHdvcmtmbG93IHVzZXMgYE9wZW5NT1NTLVRlYW0vTU9TUy1Wb2ljZUdlbmVyYXRvcmA6CgotIENyZWF0ZXMgZmljdGlvbmFsIEFtZXJpY2FuIHNwZWFrZXIgdGltYnJlcyBmcm9tIHRleHQgaW5zdHJ1Y3Rpb25zCi0gQnVpbGRzIHNwZWFrZXIgaWRlbnRpdHkgYmVmb3JlIGFnZSwgZW1vdGlvbiBhbmQgZGVsaXZlcnkgc3R5bGUKLSBEb2VzIG5vdCBnbG9iYWxseSBzbG93IGF1ZGlvIG9yIHN0cmV0Y2ggd29yZHMgdG8gaW1pdGF0ZSBhZ2UKLSBPdmVyLWdlbmVyYXRlcyBhbmQgc2NyZWVucyBjYW5kaWRhdGVzIGZvciBpZGVudGl0eSByZXBldGl0aW9uIGFuZCBhdWRpbyBxdWFsaXR5Ci0gU2F2ZXMgdGhlIHNlbGVjdGVkIGNhbmRpZGF0ZSBhcyBhIHJldXNhYmxlIENoYXR0ZXJib3ggcmVmZXJlbmNlIHZvaWNlCgojIyBPdGhlciBzdHVkaW8gZmVhdHVyZXMKCi0gQXVkaW8gMSBvbmx5IGJ5IGRlZmF1bHQ7IGFkZCByZW1vdmFibGUgQXVkaW8gMuKAkzUgd29ya3NwYWNlcwotIFNlcXVlbnRpYWwgYXVkaW8gcXVldWUgd2l0aCBHZW5lcmF0ZSBBbGwgYW5kIFF1ZXVlIE1vbml0b3IKLSBTdGFibGUgZ2VuZXJhdGVkLXZvaWNlIHByZXZpZXcgcGxheWVycwotIEZpdmUgYnVuZGxlZCBwcmVkZWZpbmVkIEFtZXJpY2FuIG1hbGUgdm9pY2VzCi0gSW1wb3J0IGFkZGl0aW9uYWwgcHJlZGVmaW5lZCBvciBjbG9uaW5nIFdBViBmaWxlcwotIE1haW4gd2F2ZWZvcm0sIGxpdmUgcGxheWhlYWQsIHpvb20sIHBhbiBhbmQgYXVkaW8gY3V0dGVyCi0gRG93bmxvYWQgZnVsbCBhdWRpbywgc2VsZWN0ZWQgYXVkaW8sIFBhcnQgT25lIG9yIFBhcnQgVHdvCi0gRmFzdEFQSSBkb2N1bWVudGF0aW9uIGF0IGAvZG9jc2AKLSBSZXNwb25zaXZlIGxpZ2h0IGFuZCBkYXJrIGludGVyZmFjZQoKIyMgUmVwb3NpdG9yeSByZWxhdGlvbnNoaXAKCmBgYHRleHQKc29mdC1tZXRhL2NoYXR0ZXJib3gtdjJAdjAuMi4xCiAgICAgICAg4oaTCnNvZnQtbWV0YS9DaGF0dGVyYm94LVRUUy1TZXJ2ZXJAdjAuOS4yCiAgICAgICAg4oaTCkNoYXR0ZXJib3ggKyBNT1NTIFZvaWNlR2VuZXJhdG9yICsgaXNvbGF0ZWQgRGl0dG8gYXZhdGFyIHdvcmtlcgpgYGAKClRoaXMgcHJvamVjdCBoYXMgbm8gcnVudGltZSBkZXBlbmRlbmN5IG9uIERldm5lbiByZXBvc2l0b3JpZXMuCgojIyBHb29nbGUgQ29sYWIgQTEwMAoKT3BlbiBgY29sYWIvU29mdE1ldGFfQ2hhdHRlcmJveF9UVFNfQ29sYWJfdjAuOS4yLmlweW5iYCwgc2VsZWN0IGFuICoqQTEwMCBHUFUqKgphbmQgcnVuIGFsbCBjZWxscy4gVGhlIG5vdGVib29rIGNyZWF0ZXMgdGhyZWUgaXNvbGF0ZWQgZW52aXJvbm1lbnRzOgoKLSBQeXRob24gMy4xMSBDaGF0dGVyYm94IHNlcnZlciBlbnZpcm9ubWVudAotIFB5dGhvbiAzLjEyIE1PU1MgVm9pY2VHZW5lcmF0b3IgZW52aXJvbm1lbnQKLSBQeXRob24gMy4xMCBEaXR0byBhdmF0YXIgZW52aXJvbm1lbnQKClRoZSBmaXJzdCBzZXR1cCBkb3dubG9hZHMgc2V2ZXJhbCBnaWdhYnl0ZXMgb2Ygdm9pY2UgYW5kIGF2YXRhciBjaGVja3BvaW50cy4KTG9uZyB2aWRlbyBnZW5lcmF0aW9uIGNhbiB0YWtlIHN1YnN0YW50aWFsIHRpbWUgYW5kIHN0b3JhZ2UgZXZlbiBvbiBBMTAwLgoKIyMgTG9jYWwgYXZhdGFyIGluc3RhbGxhdGlvbgoKQWZ0ZXIgcHJlcGFyaW5nIHRoZSBtYWluIHNlcnZlciBhbmQgbWljcm9tYW1iYSwgcnVuOgoKYGBgYmFzaApiYXNoIHNjcmlwdHMvaW5zdGFsbF9kaXR0b19hMTAwLnNoIC9wYXRoL3RvL21pY3JvbWFtYmEKYGBgCgpUaGVuIGV4cG9ydCB0aGUgcGF0aHMgcHJpbnRlZCBieSB0aGUgc2NyaXB0OgoKYGBgdGV4dApTT0ZUTUVUQV9BVkFUQVJfUFlUSE9OPS9wYXRoL3RvL2F2YXRhcjMxMC9iaW4vcHl0aG9uClNPRlRNRVRBX0RJVFRPX0RJUj0vcGF0aC90by9kaXR0by10YWxraW5naGVhZApTT0ZUTUVUQV9ESVRUT19DSEVDS1BPSU5UUz0vcGF0aC90by9kaXR0by10YWxraW5naGVhZC9jaGVja3BvaW50cwpgYGAKCiMjIERvY2tlcgoKVGhlIG1haW4gY29udGFpbmVyIGludGVudGlvbmFsbHkgZG9lcyBub3QgYmFrZSB0aGUgdmVyeSBsYXJnZSBEaXR0byBjaGVja3BvaW50cwppbnRvIHRoZSBpbWFnZS4gTW91bnQgYW4gZXh0ZXJuYWxseSBwcmVwYXJlZCBEaXR0byBkaXJlY3RvcnkgYW5kIGF2YXRhciBQeXRob24KZW52aXJvbm1lbnQsIG9yIHJ1biB0aGUgYXZhdGFyIHdvcmtlciBvbiB0aGUgR1BVIGhvc3QuIFNlZQpgZG9jcy9BVkFUQVJfVEFMS0lORy5tZGAuCgojIyBSaWdodHMgYW5kIGRpc2Nsb3N1cmUKClVzZSBvbmx5IGF2YXRhciBpbWFnZXMgYW5kIHJlZmVyZW5jZSB2b2ljZXMgdGhhdCB5b3Ugb3duIG9yIGhhdmUgcGVybWlzc2lvbiB0bwp1c2UuIERvIG5vdCBwcmVzZW50IGEgZmljdGlvbmFsIGF2YXRhciBhcyBhIHJlYWwgbmFtZWQgcGVyc29uLiBGb2xsb3cgcGxhdGZvcm0KcnVsZXMgdGhhdCByZXF1aXJlIHN5bnRoZXRpYy1tZWRpYSBkaXNjbG9zdXJlLgoKIyMgTGljZW5jZQoKU29mdE1ldGEgc2VydmVyIGFuZCBVSSBjb2RlIGFyZSBNSVQgbGljZW5zZWQuIENoYXR0ZXJib3ggaXMgTUlUIGxpY2Vuc2VkLiBNT1NTClZvaWNlR2VuZXJhdG9yIGFuZCBEaXR0byBjb2RlIGFyZSBvcGVuIHNvdXJjZSB1bmRlciB0aGVpciB1cHN0cmVhbSBsaWNlbmNlcy4KRGl0dG8ncyBwdWJsaXNoZWQgY2hlY2twb2ludCBidW5kbGUgY29udGFpbnMgdGhpcmQtcGFydHkgZmFjZS1hbmFseXNpcyBhc3NldHMKd2hvc2UgY29tbWVyY2lhbCB0ZXJtcyByZXF1aXJlIHNlcGFyYXRlIHJldmlldy4gUmVhZCBgVEhJUkRfUEFSVFlfTk9USUNFUy5tZGAKYmVmb3JlIG1vbmV0aXplZCBkZXBsb3ltZW50Lgo=","RELEASE_NOTES_V0.9.2.md":"IyBTb2Z0TWV0YSBDaGF0dGVyYm94IFRUUyBTZXJ2ZXIgdjAuOS4yCgpUaGlzIHJlbGVhc2UgZml4ZXMgdGhlIENvbGFiIEExMDAgNDBHQiBpbnN0YWxsYXRpb24gZmFpbHVyZXMgc2hvd24gaW4gdGhlIHYwLjkuMSBub3RlYm9vay4KCiMjIEZpeGVkCgotIFJlcGxhY2VkIFNwZWVjaEJyYWluIDEuMC4zIHdpdGggU3BlZWNoQnJhaW4gMS4xLjAKLSBSZW1vdmVkIHRoZSBpbmNvbXBhdGlibGUgVG9yY2hBdWRpbyBiYWNrZW5kIGltcG9ydCBmYWlsdXJlCi0gUmVtb3ZlZCB0aGUgU29mdE1ldGEgU2NpUHkgcGluIHRoYXQgY29uZmxpY3RlZCB3aXRoIHRoZSBvZmZpY2lhbCBNT1NTLVRUUyBkZXBlbmRlbmN5IHNldAotIEFkZGVkIGEgZGVkaWNhdGVkIE1PU1MgaW5zdGFsbGVyIHRoYXQgZm9sbG93cyB0aGUgdXBzdHJlYW0gYHRvcmNoLXJ1bnRpbWVgIGVudmlyb25tZW50Ci0gQWRkZWQgYHBpcCBjaGVja2AgYW5kIGltcG9ydCB2ZXJpZmljYXRpb24gZm9yIHRoZSBNT1NTIGFuZCBBdmF0YXIgZW52aXJvbm1lbnRzCi0gQWRkZWQgYSBjb21wYXRpYmlsaXR5IGd1YXJkIGZvciBzdGFsZSBjYWNoZWQgU3BlZWNoQnJhaW4gcGFja2FnZXMKLSBDaGFuZ2VkIENvbGFiIEF2YXRhciBUYWxraW5nIHRvIHRoZSBvZmZpY2lhbCBEaXR0byBQeVRvcmNoIGJhY2tlbmQgYnkgZGVmYXVsdAotIFN0b3BwZWQgYXV0b21hdGljYWxseSBidWlsZGluZyBsZWdhY3kgVGVuc29yUlQgOC42LjEgb24gY3VycmVudCBDb2xhYiBpbWFnZXMKLSBQcmV2ZW50ZWQgdGhlIHNlcnZlciBmcm9tIHNlbGVjdGluZyBUZW5zb3JSVCB3aGVuIHRoZSBQeXRob24gcnVudGltZSBjYW5ub3QgaW1wb3J0IGl0Ci0gUHJlc2VydmVkIHRoZSBBMTAwIDQwR0IgY2hlY2twb2ludGVkIGxvbmctdmlkZW8gcHJvZmlsZSBmb3IgMTAsIDIwIGFuZCAzMC1taW51dGUgcmVuZGVycwoKIyMgUnVudGltZSBwcm9maWxlCgotIEdQVTogQ29sYWIgTlZJRElBIEExMDAgNDBHQgotIFRUUzogQ2hhdHRlcmJveCBwbHVzIGlzb2xhdGVkIE1PU1MgVm9pY2VHZW5lcmF0b3IKLSBBdmF0YXI6IERpdHRvIFB5VG9yY2gKLSBMb25nIHZpZGVvOiBjaGVja3BvaW50ZWQgbW9kZQotIFdvcmtpbmcgcmVzb2x1dGlvbjogcmVkdWNlZCBpbnRlcm5hbGx5IGZvciBzdGFiaWxpdHkKLSBFeHBvcnQ6IDcyMHAgb3IgMTA4MHAKCiMjIFJlcXVpcmVkIGVuZ2luZQoKYHNvZnQtbWV0YS9jaGF0dGVyYm94LXYyQHYwLjIuMWAKClJ1biBgY29sYWIvU29mdE1ldGFfQ2hhdHRlcmJveF9UVFNfQ29sYWJfdjAuOS4yLmlweW5iYCBpbiBhIGZyZXNoIEExMDAgcnVudGltZS4K","avatar_engine.py":"...","docs/AVATAR_TALKING.md":"IyBBdmF0YXIgVGFsa2luZyBpbiBTb2Z0TWV0YSB2MC45LjIKCiMjIFVJIHBsYWNlbWVudAoKYEdlbmVyYXRlIFZpZGVvYCBpcyBub3QgY291bnRlZCBhcyBvbmUgb2YgdGhlIGZpdmUgYXVkaW8gd29ya3NwYWNlcy4gVGhlIGJyb3dzZXIKcmVidWlsZHMgdGhlIHRhYiByb3cgaW4gdGhpcyBvcmRlcjoKCjEuIEV2ZXJ5IGV4aXN0aW5nIEF1ZGlvIHdvcmtzcGFjZQoyLiBHZW5lcmF0ZSBWaWRlbwozLiBUaGUgcGx1cyBidXR0b24sIHdoaWxlIGZld2VyIHRoYW4gZml2ZSBBdWRpbyB3b3Jrc3BhY2VzIGV4aXN0CgpUaGlzIGd1YXJhbnRlZXMgdGhhdCBhZGRpbmcgQXVkaW8gMuKAkzUgbW92ZXMgR2VuZXJhdGUgVmlkZW8gdG8gdGhlIHJpZ2h0LgoKIyMgUmVuZGVyaW5nIGZsb3cKCmBgYHRleHQKQXZhdGFyIGltYWdlICsgY29tcGxldGVkIG9yIHVwbG9hZGVkIGF1ZGlvCiAgLT4gaW1hZ2Ugbm9ybWFsaXphdGlvbiBhbmQgYXNwZWN0LXJhdGlvIGZyYW1pbmcKICAtPiBtb25vIDE2IGtIeiBzcGVlY2ggcHJlcGFyYXRpb24gZm9yIERpdHRvCiAgLT4gY29udGludW91cyBvciBzaWxlbmNlLWF3YXJlIGNoZWNrcG9pbnRlZCByZW5kZXJpbmcKICAtPiBzZWN0aW9uIGpvaW5pbmcKICAtPiBvcmlnaW5hbCBhdWRpbyByZXN0b3JhdGlvbgogIC0+IEguMjY0L0FBQyBmaW5hbCBlbmNvZGluZwogIC0+IGR1cmF0aW9uIGFuZCBmcmVlemUgcXVhbGl0eSBjaGVja3MKYGBgCgpDb250aW51b3VzIG1vZGUgc2VuZHMgdGhlIHdob2xlIGF1ZGlvIHRvIERpdHRvIGFuZCBnaXZlcyB0aGUgYmVzdCBtb3Rpb24KY29udGludWl0eS4gQ2hlY2twb2ludGVkIG1vZGUgaXMgbW9yZSByZXNpbGllbnQgZm9yIHZlcnkgbG9uZyBqb2JzLiBJdCBzZWFyY2hlcwpmb3IgbmF0dXJhbCBzaWxlbmNlIG5lYXIgdGhlIHNlbGVjdGVkIGNoZWNrcG9pbnQgbGVuZ3RoIGFuZCByZW5kZXJzIGVhY2ggc2VjdGlvbgppbmRlcGVuZGVudGx5LiBCZWNhdXNlIGEgc3RpbGwgaW1hZ2UgaW5pdGlhbGl6ZXMgZWFjaCBzZWN0aW9uLCBpbnNwZWN0IHRoZSBqb2lucwpiZWZvcmUgcHVibGlzaGluZy4KCiMjIEFQSQoKLSBgR0VUIC9hcGkvdmlkZW8vc3RhdHVzYAotIGBQT1NUIC9hcGkvdmlkZW8vYXZhdGFyLXVwbG9hZGAKLSBgR0VUIC9hcGkvdmlkZW8vYXZhdGFyL3tmaWxlbmFtZX1gCi0gYFBPU1QgL2FwaS92aWRlby9hdWRpby11cGxvYWRgCi0gYEdFVCAvYXBpL3ZpZGVvL2F1ZGlvL3tmaWxlbmFtZX1gCi0gYEdFVCAvYXBpL3ZpZGVvL2pvYnNgCi0gYFBPU1QgL2FwaS92aWRlby9qb2JzYAotIGBHRVQgL2FwaS92aWRlby9qb2JzL3tqb2JfaWR9YAotIGBQT1NUIC9hcGkvdmlkZW8vam9icy97am9iX2lkfS9jYW5jZWxgCi0gYERFTEVURSAvYXBpL3ZpZGVvL2pvYnMve2pvYl9pZH1gCi0gYERFTEVURSAvYXBpL3ZpZGVvL2pvYnNgCi0gYEdFVCAvYXBpL3ZpZGVvL2pvYnMve2pvYl9pZH0vbG9nYAotIGBHRVQgL2FwaS92aWRlby9qb2JzL3tqb2JfaWR9L2ZpbGVgCgojIyBTdG9yYWdlCgpgYGB0ZXh0CmRhdGEvYXZhdGFyX2ltYWdlcyAgIHVwbG9hZGVkIGF2YXRhciBpbWFnZXMKZGF0YS92aWRlb19hdWRpbyAgICAgc2VwYXJhdGVseSB1cGxvYWRlZCBhdWRpbwpkYXRhL3ZpZGVvX3dvcmsgICAgICB0ZW1wb3JhcnkgcmVuZGVyIHNlY3Rpb25zCnZpZGVvX291dHB1dHMgICAgICAgIGNvbXBsZXRlZCBNUDQgZmlsZXMKbG9ncy9hdmF0YXJfKi5sb2cgICAgcGVyLWpvYiBlbmdpbmUgb3V0cHV0CmBgYAoKIyMgQTEwMCBkZWZhdWx0cwoKLSBFbmdpbmU6IEF1dG8sIHByZWZlcnJpbmcgRGl0dG8gVGVuc29yUlQKLSBGYWxsYmFjazogRGl0dG8gUHlUb3JjaAotIE1vZGU6IENvbnRpbnVvdXMKLSBGaW5hbCBkZWxpdmVyeTogMTA4MHAgcG9ydHJhaXQsIDI1IGZwcywgSC4yNjQvQUFDCi0gQ2hlY2twb2ludCBsZW5ndGg6IGFib3V0IDMgbWludXRlcyB3aGVuIGNoZWNrcG9pbnRlZCBtb2RlIGlzIHNlbGVjdGVkCgpUaGUgZW5naW5lIGlzIGlzb2xhdGVkIGZyb20gQ2hhdHRlcmJveCBhbmQgTU9TUyBiZWNhdXNlIHRoZWlyIENVREEsIFB5VG9yY2ggYW5kClRyYW5zZm9ybWVycyByZXF1aXJlbWVudHMgZGlmZmVyLgoKIyMgUmVhbGlzbSBndWlkYW5jZQoKVXNlIG9uZSBjbGVhciBwZXJzb24sIGEgbW9zdGx5IGZvcndhcmQgZmFjZSwgdmlzaWJsZSBleWVzIGFuZCBtb3V0aCwgbmF0dXJhbCByb29tCmxpZ2h0aW5nIGFuZCBlbm91Z2ggcmVzb2x1dGlvbi4gVXBwZXItYm9keSBwb3J0cmFpdHMgdXN1YWxseSBnaXZlIGJldHRlcgpyZXN1bHRzIHRoYW4gZGlzdGFudCBmdWxsLWJvZHkgaW1hZ2VzLiBUaGUgc3lzdGVtIGNhbiBkZXRlY3QgdGVjaG5pY2FsIGZhaWx1cmVzLApidXQgaXQgY2Fubm90IHJlbGlhYmx5IGRldGVjdCBldmVyeSB1bm5hdHVyYWwgYmxpbmssIHRvb3RoIHNoYXBlLCBsaXAgc2hhcGUgb3IKYmFja2dyb3VuZCBkZWZvcm1hdGlvbi4gV2F0Y2ggdGhlIGNvbXBsZXRlIHJlc3VsdC4KCiMjIENvbW1lcmNpYWwgZGVwbG95bWVudCB3YXJuaW5nCgpEaXR0byBjb2RlIGlzIEFwYWNoZS0yLjAuIEl0cyBvZmZpY2lhbCBjaGVja3BvaW50IGJ1bmRsZSBpbmNsdWRlcyBmYWNlLWRldGVjdGlvbgphc3NldHMgbmFtZWQgYGluc2lnaHRmYWNlX2RldGAgb3IgYGRldF8xMGdgLiBJbnNpZ2h0RmFjZSBzdGF0ZXMgdGhhdCBpdHMgc3VwcGxpZWQKcHJldHJhaW5lZCBtb2RlbHMgYXJlIGZvciBub24tY29tbWVyY2lhbCByZXNlYXJjaCB1bmxlc3Mgc2VwYXJhdGVseSBsaWNlbnNlZC4KU29mdE1ldGEgZG9lcyBub3QgY2hhbmdlIG9yIGdyYW50IHRob3NlIHJpZ2h0cy4gT2J0YWluIGFwcHJvcHJpYXRlIHBlcm1pc3Npb24gb3IKcmVwbGFjZSB0aGUgcmVzdHJpY3RlZCBkZXRlY3RvciBhc3NldHMgYmVmb3JlIG1vbmV0aXplZCBkZXBsb3ltZW50LgoKIyMgdjAuOS4yIENvbGFiIGJhY2tlbmQgcG9saWN5CgpDdXJyZW50IENvbGFiIEExMDAgaW1hZ2VzIHVzZSBhIG5ld2VyIENVREEvUHl0aG9uIHN0YWNrIHRoYW4gdGhlIGxlZ2FjeQpUZW5zb3JSVCA4LjYuMSB3aGVlbCB1c2VkIGJ5IHRoZSBvcmlnaW5hbCBEaXR0byB0ZXN0IGVudmlyb25tZW50LiBTb2Z0TWV0YQp0aGVyZWZvcmUgaW5zdGFsbHMgYW5kIHNlbGVjdHMgdGhlIG9mZmljaWFsIERpdHRvIFB5VG9yY2ggY2hlY2twb2ludCBieQpkZWZhdWx0LiBUaGlzIGF2b2lkcyBhIGZhbHNlIGluc3RhbGxhdGlvbiBmYWlsdXJlIGFuZCBrZWVwcyBvdXRwdXQgcXVhbGl0eQp1bmNoYW5nZWQuIFRlbnNvclJUIHJlbWFpbnMgYW4gb3B0LWluIGFkdmFuY2VkIHBhdGggZm9yIG1hdGNoaW5nIGN1c3RvbSBpbWFnZXMuCgpTZXQgYFNPRlRNRVRBX1RSWV9URU5TT1JSVD0xYCBkdXJpbmcgaW5zdGFsbGF0aW9uIGFuZApgU09GVE1FVEFfRU5BQkxFX1RFTlNPUlJUPTFgIGF0IHJ1bnRpbWUgb25seSB3aGVuIFRlbnNvclJUIGltcG9ydHMgc3VjY2Vzc2Z1bGx5Lgo=","requirements-voice.txt":"IyBBZGQtb25zIGZvciB0aGUgaXNvbGF0ZWQgTU9TUyBWb2ljZUdlbmVyYXRvciBlbnZpcm9ubWVudC4KIyBNT1NTLVRUUyBvd25zIGl0cyBvd24gdG9yY2gsIHRvcmNoYXVkaW8sIHRyYW5zZm9ybWVycyBhbmQgc2NpcHkgdmVyc2lvbnMKIyB0aHJvdWdoIGl0cyBvZmZpY2lhbCBbdG9yY2gtcnVudGltZV0gZXh0cmEuIERvIG5vdCBvdmVycmlkZSB0aGVtIGhlcmUuCnNwZWVjaGJyYWluPT0xLjEuMApzb3VuZGZpbGU9PTAuMTMuMQpodWdnaW5nZmFjZS1odWI+PTAuMzYuMApzZXR1cHRvb2xzPT04MC45LjAK","scripts/install_ditto_a100.sh":"XAojIS91c3IvYmluL2VudiBiYXNoCnNldCAtZXVvIHBpcGVmYWlsCgojIFNvZnRNZXRhIHYwLjkuMiBBMTAwIDQwR0IgRGl0dG8gaW5zdGFsbGVyLgojIFN0YWJsZSBkZWZhdWx0OiBvZmZpY2lhbCBEaXR0byBQeVRvcmNoIGJhY2tlbmQuCiMgVGVuc29yUlQgOC42LjEgaXMgbm90IGluc3RhbGxlZCBhdXRvbWF0aWNhbGx5IGJlY2F1c2UgY3VycmVudCBDb2xhYiBDVURBIC8KIyBQeXRob24gaW1hZ2VzIG9mdGVuIGNhbm5vdCBidWlsZCB0aGF0IGxlZ2FjeSBwYWNrYWdlLiBTZXQKIyBTT0ZUTUVUQV9UUllfVEVOU09SUlQ9MSBvbmx5IHdoZW4gdXNpbmcgYSBtYXRjaGluZyBDVURBL1RlbnNvclJUIGltYWdlLgoKTU09IiR7MTotL2NvbnRlbnQvYmluL21pY3JvbWFtYmF9IgpFTlZfTkFNRT0iJHtTT0ZUTUVUQV9BVkFUQVJfRU5WOi1hdmF0YXIzMTB9IgpESVRUT19ESVI9IiR7U09GVE1FVEFfRElUVE9fRElSOi0vY29udGVudC9kaXR0by10YWxraW5naGVhZH0iClNFUlZFUl9ESVI9IiR7U09GVE1FVEFfU0VSVkVSX0RJUjotL2NvbnRlbnQvQ2hhdHRlcmJveC1UVFMtU2VydmVyfSIKVFJZX1RFTlNPUlJUPSIke1NPRlRNRVRBX1RSWV9URU5TT1JSVDotMH0iCgppZiBbWyAhIC14ICIkTU0iIF1dOyB0aGVuCiAgZWNobyAibWljcm9tYW1iYSB3YXMgbm90IGZvdW5kIGF0OiAkTU0iID4mMgogIGV4aXQgMgpmaQppZiBbWyAhIC1mICIkU0VSVkVSX0RJUi9yZXF1aXJlbWVudHMtYXZhdGFyLnR4dCIgXV07IHRoZW4KICBlY2hvICJTb2Z0TWV0YSBzZXJ2ZXIgd2FzIG5vdCBmb3VuZCBhdDogJFNFUlZFUl9ESVIiID4mMgogIGV4aXQgMgpmaQoKaWYgIiRNTSIgZW52IGxpc3QgfCBhd2sgJ3twcmludCAkMX0nIHwgZ3JlcCAtcXggIiRFTlZfTkFNRSI7IHRoZW4KICAiJE1NIiBlbnYgcmVtb3ZlIC1uICIkRU5WX05BTUUiIC15IHx8IHRydWUKZmkKIiRNTSIgY3JlYXRlIC15IC1uICIkRU5WX05BTUUiIC1jIGNvbmRhLWZvcmdlIHB5dGhvbj0zLjEwIHBpcAoKaWYgY29tbWFuZCAtdiBudmlkaWEtc21pID4vZGV2L251bGwgMj4mMTsgdGhlbgogIGVjaG8gIkRldGVjdGVkIEdQVToiCiAgbnZpZGlhLXNtaSAtLXF1ZXJ5LWdwdT1uYW1lLG1lbW9yeS50b3RhbCAtLWZvcm1hdD1jc3Ysbm9oZWFkZXIKZmkKCnJtIC1yZiAiJERJVFRPX0RJUiIKZ2l0IGNsb25lIC0tZGVwdGggMSBodHRwczovL2dpdGh1Yi5jb20vYW50Z3JvdXAvZGl0dG8tdGFsa2luZ2hlYWQuZ2l0ICIkRElUVE9fRElSIgoKIiRNTSIgcnVuIC1uICIkRU5WX05BTUUiIHB5dGhvbiAtbSBwaXAgaW5zdGFsbCAtVSBwaXAgd2hlZWwgc2V0dXB0b29scwoiJE1NIiBydW4gLW4gIiRFTlZfTkFNRSIgcHl0aG9uIC1tIHBpcCBpbnN0YWxsIFwKICAtLWluZGV4LXVybCBodHRwczovL2Rvd25sb2FkLnB5dG9yY2gub3JnL3dobC9jdTEyMSBcCiAgdG9yY2g9PTIuNS4xIHRvcmNodmlzaW9uPT0wLjIwLjEgdG9yY2hhdWRpbz09Mi41LjEKIiRNTSIgcnVuIC1uICIkRU5WX05BTUUiIHB5dGhvbiAtbSBwaXAgaW5zdGFsbCAtLW5vLWNhY2hlLWRpciBcCiAgLXIgIiRTRVJWRVJfRElSL3JlcXVpcmVtZW50cy1hdmF0YXIudHh0IgoKaWYgW1sgIiRUUllfVEVOU09SUlQiID09ICIxIiBdXTsgdGhlbgogIGVjaG8gIk9wdGlvbmFsIFRlbnNvclJUIGluc3RhbGxhdGlvbiByZXF1ZXN0ZWQuIgogIHNldCArZQogICIkTU0iIHJ1biAtbiAiJEVOVl9OQU1FIiBweXRob24gLW0gcGlwIGluc3RhbGwgLS1uby1jYWNoZS1kaXIgXAogICAgLS1leHRyYS1pbmRleC11cmwgaHR0cHM6Ly9weXBpLm52aWRpYS5jb20gXAogICAgdGVuc29ycnQ9PTguNi4xCiAgVFJUX1NUQVRVUz0kPwogIHNldCAtZQogIGlmIFtbICRUUlRfU1RBVFVTIC1uZSAwIF1dOyB0aGVuCiAgICBlY2hvICJPcHRpb25hbCBUZW5zb3JSVCBpbnN0YWxsYXRpb24gZmFpbGVkLiBEaXR0byBQeVRvcmNoIHJlbWFpbnMgYXZhaWxhYmxlLiIKICBmaQplbHNlCiAgZWNobyAiU2tpcHBpbmcgbGVnYWN5IFRlbnNvclJUIDguNi4xIG9uIENvbGFiLiBEaXR0byBQeVRvcmNoIGlzIHRoZSBzdGFibGUgQTEwMCA0MEdCIGJhY2tlbmQuIgpmaQoKIiRNTSIgcnVuIC1uICIkRU5WX05BTUUiIHB5dGhvbiAtIDw8UFlET1dOTE9BRApmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgc25hcHNob3RfZG93bmxvYWQKc25hcHNob3RfZG93bmxvYWQoCiAgICByZXBvX2lkPSJkaWdpdGFsLWF2YXRhci9kaXR0by10YWxraW5naGVhZCIsCiAgICBsb2NhbF9kaXI9ciIkRElUVE9fRElSL2NoZWNrcG9pbnRzIiwKKQpwcmludCgiRGl0dG8gY2hlY2twb2ludHMgZG93bmxvYWRlZC4iKQpQWURPV05MT0FECgoiJE1NIiBydW4gLW4gIiRFTlZfTkFNRSIgcHl0aG9uIC1tIHBpcCBjaGVjawoKQVZBVEFSX1BZVEhPTj0iJCgkTU0gcnVuIC1uICIkRU5WX05BTUUiIHB5dGhvbiAtYyAnaW1wb3J0IHN5czsgcHJpbnQoc3lzLmV4ZWN1dGFibGUpJykiCiIkTU0iIHJ1biAtbiAiJEVOVl9OQU1FIiBweXRob24gLSA8PFBZVkVSSUZZCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgdG9yY2gKcm9vdCA9IFBhdGgociIkRElUVE9fRElSIikKcmVxdWlyZWQgPSBbCiAgICByb290IC8gImluZmVyZW5jZS5weSIsCiAgICByb290IC8gImNoZWNrcG9pbnRzL2RpdHRvX2NmZy92MC40X2h1YmVydF9jZmdfcHl0b3JjaC5wa2wiLAogICAgcm9vdCAvICJjaGVja3BvaW50cy9kaXR0b19weXRvcmNoIiwKXQptaXNzaW5nID0gW3N0cihwYXRoKSBmb3IgcGF0aCBpbiByZXF1aXJlZCBpZiBub3QgcGF0aC5leGlzdHMoKV0KcHJpbnQoIlB5VG9yY2g6IiwgdG9yY2guX192ZXJzaW9uX18pCnByaW50KCJDVURBIGF2YWlsYWJsZToiLCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpKQppZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgcHJpbnQoIkdQVToiLCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSkKICAgIHByaW50KCJHUFUgVlJBTSBHQjoiLCByb3VuZCh0b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcygwKS50b3RhbF9tZW1vcnkgLyAoMTAyNCAqKiAzKSwgMSkpCmlmIG1pc3Npbmc6CiAgICByYWlzZSBTeXN0ZW1FeGl0KCJNaXNzaW5nIERpdHRvIGZpbGVzOiAiICsgIiwgIi5qb2luKG1pc3NpbmcpKQpwcmludCgiRGl0dG8gUHlUb3JjaCBiYWNrZW5kIGlzIHJlYWR5LiIpCnRyeToKICAgIGltcG9ydCB0ZW5zb3JydApleGNlcHQgRXhjZXB0aW9uOgogICAgcHJpbnQoIlRlbnNvclJUOiBub3QgaW5zdGFsbGVkLCBQeVRvcmNoIG1vZGUgc2VsZWN0ZWQuIikKZWxzZToKICAgIHByaW50KCJUZW5zb3JSVDoiLCB0ZW5zb3JydC5fX3ZlcnNpb25fXykKUFlWRVJJRlkKCmNhdCA8PFNVTU1BUlkKCkF2YXRhciBpbnN0YWxsYXRpb24gY29tcGxldGVkLgpTdGFibGUgYmFja2VuZDogRGl0dG8gUHlUb3JjaApTZXQgdGhlc2UgdmFyaWFibGVzIGJlZm9yZSBzdGFydGluZyBTb2Z0TWV0YToKICBTT0ZUTUVUQV9BVkFUQVJfUFlUSE9OPSRBVkFUQVJfUFlUSE9OCiAgU09GVE1FVEFfRElUVE9fRElSPSRESVRUT19ESVIKICBTT0ZUTUVUQV9ESVRUT19DSEVDS1BPSU5UUz0kRElUVE9fRElSL2NoZWNrcG9pbnRzCiAgU09GVE1FVEFfRU5BQkxFX1RFTlNPUlJUPTAKCkltcG9ydGFudDogcmV2aWV3IFRISVJEX1BBUlRZX05PVElDRVMubWQgYmVmb3JlIGNvbW1lcmNpYWwgZGVwbG95bWVudC4KU1VNTUFSWQo=","scripts/install_moss_a100.sh":"XAojIS91c3IvYmluL2VudiBiYXNoCnNldCAtZXVvIHBpcGVmYWlsCgojIFNvZnRNZXRhIHYwLjkuMiBNT1NTIFZvaWNlR2VuZXJhdG9yIGluc3RhbGxlci4KIyBVc2VzIHRoZSB1cHN0cmVhbSBNT1NTLVRUUyBkZXBlbmRlbmN5IHNldCBpbnN0ZWFkIG9mIG92ZXJyaWRpbmcgdG9yY2gsCiMgdG9yY2hhdWRpbywgdHJhbnNmb3JtZXJzLCBvciBzY2lweSB3aXRoIGNvbmZsaWN0aW5nIHZlcnNpb25zLgoKTU09IiR7MTotL2NvbnRlbnQvYmluL21pY3JvbWFtYmF9IgpFTlZfTkFNRT0iJHtTT0ZUTUVUQV9NT1NTX0VOVjotbW9zczMxMn0iCk1PU1NfRElSPSIke1NPRlRNRVRBX01PU1NfRElSOi0vY29udGVudC9NT1NTLVRUU30iClNFUlZFUl9ESVI9IiR7U09GVE1FVEFfU0VSVkVSX0RJUjotL2NvbnRlbnQvQ2hhdHRlcmJveC1UVFMtU2VydmVyfSIKCmlmIFtbICEgLXggIiRNTSIgXV07IHRoZW4KICBlY2hvICJtaWNyb21hbWJhIHdhcyBub3QgZm91bmQgYXQ6ICRNTSIgPiYyCiAgZXhpdCAyCmZpCmlmIFtbICEgLWYgIiRTRVJWRVJfRElSL3JlcXVpcmVtZW50cy12b2ljZS50eHQiIF1dOyB0aGVuCiAgZWNobyAiU29mdE1ldGEgc2VydmVyIHdhcyBub3QgZm91bmQgYXQ6ICRTRVJWRVJfRElSIiA+JjIKICBleGl0IDIKZmkKCmlmICIkTU0iIGVudiBsaXN0IHwgYXdrICd7cHJpbnQgJDF9JyB8IGdyZXAgLXF4ICIkRU5WX05BTUUiOyB0aGVuCiAgIiRNTSIgZW52IHJlbW92ZSAtbiAiJEVOVl9OQU1FIiAteSB8fCB0cnVlCmZpCiIkTU0iIGNyZWF0ZSAteSAtbiAiJEVOVl9OQU1FIiAtYyBjb25kYS1mb3JnZSBweXRob249My4xMiBwaXAKCnJtIC1yZiAiJE1PU1NfRElSIgpnaXQgY2xvbmUgLS1kZXB0aCAxIGh0dHBzOi8vZ2l0aHViLmNvbS9PcGVuTU9TUy9NT1NTLVRUUy5naXQgIiRNT1NTX0RJUiIKCiIkTU0iIHJ1biAtbiAiJEVOVl9OQU1FIiBweXRob24gLW0gcGlwIGluc3RhbGwgLVUgcGlwIHdoZWVsIHNldHVwdG9vbHMKIiRNTSIgcnVuIC1uICIkRU5WX05BTUUiIHB5dGhvbiAtbSBwaXAgaW5zdGFsbCAtLW5vLWNhY2hlLWRpciBcCiAgLS1leHRyYS1pbmRleC11cmwgaHR0cHM6Ly9kb3dubG9hZC5weXRvcmNoLm9yZy93aGwvY3UxMjggXAogIC1lICIkTU9TU19ESVJbdG9yY2gtcnVudGltZV0iCiIkTU0iIHJ1biAtbiAiJEVOVl9OQU1FIiBweXRob24gLW0gcGlwIGluc3RhbGwgLS1uby1jYWNoZS1kaXIgXAogIC1yICIkU0VSVkVSX0RJUi9yZXF1aXJlbWVudHMtdm9pY2UudHh0IgoKIiRNTSIgcnVuIC1uICIkRU5WX05BTUUiIHB5dGhvbiAtbSBwaXAgY2hlY2sKCiIkTU0iIHJ1biAtbiAiJEVOVl9OQU1FIiBweXRob24gLSA8PCdQWVZFUklGWScKaW1wb3J0IHN5cwpmcm9tIGltcG9ydGxpYi5tZXRhZGF0YSBpbXBvcnQgdmVyc2lvbgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoYXVkaW8KaW1wb3J0IHNvdW5kZmlsZQpmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsLCBBdXRvUHJvY2Vzc29yCmZyb20gc3BlZWNoYnJhaW4uaW5mZXJlbmNlLnNwZWFrZXIgaW1wb3J0IEVuY29kZXJDbGFzc2lmaWVyCgpwcmludCgiTU9TUyBWb2ljZUdlbmVyYXRvciBlbnZpcm9ubWVudCIpCnByaW50KCJQeXRob246Iiwgc3lzLnZlcnNpb24pCnByaW50KCJQeVRvcmNoOiIsIHRvcmNoLl9fdmVyc2lvbl9fKQpwcmludCgiVG9yY2hBdWRpbzoiLCB0b3JjaGF1ZGlvLl9fdmVyc2lvbl9fKQpwcmludCgiVHJhbnNmb3JtZXJzOiIsIHZlcnNpb24oInRyYW5zZm9ybWVycyIpKQpwcmludCgiU3BlZWNoQnJhaW46IiwgdmVyc2lvbigic3BlZWNoYnJhaW4iKSkKcHJpbnQoIlNvdW5kRmlsZToiLCBzb3VuZGZpbGUuX192ZXJzaW9uX18pCnByaW50KCJDVURBIGF2YWlsYWJsZToiLCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpKQppZiBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgIHJhaXNlIFN5c3RlbUV4aXQoIkNVREEgaXMgdW5hdmFpbGFibGUgaW4gdGhlIE1PU1MgZW52aXJvbm1lbnQuIikKcHJpbnQoIkdQVToiLCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSkKcHJpbnQoIkF1dG9Nb2RlbDoiLCBBdXRvTW9kZWwuX19uYW1lX18pCnByaW50KCJBdXRvUHJvY2Vzc29yOiIsIEF1dG9Qcm9jZXNzb3IuX19uYW1lX18pCnByaW50KCJTcGVha2VyIGNoZWNrZXI6IiwgRW5jb2RlckNsYXNzaWZpZXIuX19uYW1lX18pCnByaW50KCJNT1NTIGVudmlyb25tZW50IHZlcmlmaWNhdGlvbiBwYXNzZWQuIikKUFlWRVJJRlkKCmVjaG8gIk1PU1MgVm9pY2VHZW5lcmF0b3IgaW5zdGFsbGF0aW9uIGNvbXBsZXRlZCB3aXRob3V0IGRlcGVuZGVuY3kgY29uZmxpY3RzLiIK","server.py":"...","ui/app.js":"...","ui/index.html":"...","ui/styles.css":"...","voice_worker.py":"..."}
for relative, payload in files.items():
    destination = project / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(payload))
for script in project.glob("scripts/*.sh"):
    script.chmod(0o755)
print("Applied embedded SoftMeta v0.9.2 patch:", len(files), "files")
SOFTMETA_PATCH

# Main Chatterbox environment.
"$MM" run -n sm311 python -m pip install -U pip wheel
"$MM" run -n sm311 python -m pip install "setuptools==80.9.0"
"$MM" run -n sm311 python -m pip install   --index-url https://download.pytorch.org/whl/cu124   torch==2.6.0 torchaudio==2.6.0
"$MM" run -n sm311 python -m pip install --no-cache-dir chatterbox-tts==0.1.7
"$MM" run -n sm311 python -m pip install --force-reinstall "setuptools==80.9.0"
"$MM" run -n sm311 python -m pip install --no-deps -e /content/chatterbox-v2
"$MM" run -n sm311 python -m pip install   -r /content/Chatterbox-TTS-Server/requirements-colab.txt
"$MM" run -n sm311 python -m pip install --force-reinstall "setuptools==80.9.0"
# Official MOSS dependency installation. This script does not override the
# upstream torch, torchaudio, transformers, or scipy versions.
export SOFTMETA_SERVER_DIR=/content/Chatterbox-TTS-Server
bash /content/Chatterbox-TTS-Server/scripts/install_moss_a100.sh "$MM"

echo "Chatterbox and MOSS installations completed."


## Install the A100 Avatar Talking worker

This installs the official Ditto PyTorch checkpoint in an isolated Python 3.10
environment. It intentionally skips legacy TensorRT 8.6.1 on normal Colab A100
images, preventing the failed wheel-build message seen in v0.9.1.


In [ ]:
%%bash
set -euo pipefail

export SOFTMETA_SERVER_DIR=/content/Chatterbox-TTS-Server
export SOFTMETA_DITTO_DIR=/content/ditto-talkinghead
export SOFTMETA_AVATAR_ENV=avatar310
export SOFTMETA_TRY_TENSORRT=0

bash /content/Chatterbox-TTS-Server/scripts/install_ditto_a100.sh /content/bin/micromamba


In [ ]:
%%bash
set -euo pipefail
MM="/content/bin/micromamba"

"$MM" run -n sm311 python - <<'PYMAIN'
import sys
from importlib.metadata import version
import torch
import chatterbox
import perth
from softmeta_chatterbox import SoftMetaChatterboxEngine

print("Main Chatterbox environment")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", version("transformers"))
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("CUDA is unavailable. Change the runtime to an A100 GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("Official Chatterbox package:", chatterbox.__file__)
print("PerTh watermarker callable:", callable(getattr(perth, "PerthImplicitWatermarker", None)))
runtime = SoftMetaChatterboxEngine(device="auto")
print("SoftMeta engine device:", runtime.device)
PYMAIN

"$MM" run -n moss312 python - <<'PYMOSS'
import sys
from importlib.metadata import version
import torch
import torchaudio
import soundfile
from transformers import AutoModel, AutoProcessor
from speechbrain.inference.speaker import EncoderClassifier

print()
print("MOSS VoiceGenerator environment")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)
print("Transformers:", version("transformers"))
print("SpeechBrain:", version("speechbrain"))
print("SoundFile:", soundfile.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("CUDA is unavailable in the MOSS environment.")
print("GPU:", torch.cuda.get_device_name(0))
print("AutoModel:", AutoModel.__name__)
print("AutoProcessor:", AutoProcessor.__name__)
print("Speaker checker:", EncoderClassifier.__name__)
print("MOSS verification passed.")
PYMOSS

"$MM" run -n avatar310 python - <<'PYAVATAR'
from pathlib import Path
import torch

root = Path('/content/ditto-talkinghead')
print()
print("Avatar Talking environment")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("CUDA is unavailable in the Avatar environment.")
name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print("GPU:", name)
print("GPU VRAM GB:", round(total_gb, 1))
if total_gb < 38:
    raise SystemExit("Avatar Talking requires an A100-class GPU with about 40GB VRAM.")
if total_gb < 48:
    print("A100 40GB profile enabled: checkpointed long-video rendering and reduced working resolution.")

required = [
    root / 'inference.py',
    root / 'checkpoints/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl',
    root / 'checkpoints/ditto_pytorch',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise SystemExit('Missing Ditto files: ' + ', '.join(missing))
print("Ditto PyTorch backend is ready.")
print("TensorRT is disabled by default for Colab compatibility.")
print("Avatar Talking verification passed.")
PYAVATAR

echo
echo "All three isolated environments passed verification."


In [ ]:
import os
import signal
import socket
import subprocess
import time
from pathlib import Path
from IPython.display import HTML, display

PORT = 8004
PROJECT = Path('/content/Chatterbox-TTS-Server')
LOG = Path('/content/softmeta_chatterbox_v092.log')
PID_FILE = Path('/content/softmeta_chatterbox_v092.pid')
MM = '/content/bin/micromamba'

if PID_FILE.exists():
    try:
        os.kill(int(PID_FILE.read_text().strip()), signal.SIGTERM)
        time.sleep(1)
    except Exception:
        pass
subprocess.run(f"lsof -t -i:{PORT} | xargs -r kill -9", shell=True, check=False)
LOG.unlink(missing_ok=True)

def env_python(name: str) -> str:
    return subprocess.check_output(
        [MM, 'run', '-n', name, 'python', '-c', 'import sys; print(sys.executable)'],
        text=True,
    ).strip()

env = {
    **os.environ,
    'PYTHONUNBUFFERED': '1',
    'HF_HOME': '/content/hf_home',
    'HF_HUB_CACHE': '/content/hf_home/hub',
    'TRANSFORMERS_CACHE': '/content/hf_home/transformers',
    'SOFTMETA_DEVICE': 'cuda',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True,max_split_size_mb:128',
    'SOFTMETA_AVATAR_GPU_PROFILE': 'a100_40gb',
    'SOFTMETA_MODEL': 'chatterbox',
    'SOFTMETA_VOICE_PYTHON': env_python('moss312'),
    'SOFTMETA_MOSS_MODEL_DIR': '/content/softmeta_models/moss_voice_generator',
    'SOFTMETA_AVATAR_PYTHON': env_python('avatar310'),
    'SOFTMETA_DITTO_DIR': '/content/ditto-talkinghead',
    'SOFTMETA_DITTO_CHECKPOINTS': '/content/ditto-talkinghead/checkpoints',
    'SOFTMETA_ENABLE_TENSORRT': '0',
}
Path(env['HF_HOME']).mkdir(parents=True, exist_ok=True)

log_handle = LOG.open('w', encoding='utf-8', errors='replace')
process = subprocess.Popen(
    [MM, 'run', '-n', 'sm311', 'python', '-u', 'start.py'],
    cwd=PROJECT,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
PID_FILE.write_text(str(process.pid), encoding='utf-8')

def port_open() -> bool:
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=.5):
            return True
    except OSError:
        return False

print('Starting SoftMeta Chatterbox TTS Server v0.9.2...')
for _ in range(360):
    if process.poll() is not None:
        log_handle.flush()
        raise RuntimeError(LOG.read_text(errors='replace')[-20000:])
    if port_open():
        break
    time.sleep(1)
else:
    raise TimeoutError('The server did not open port 8004. Run the log cell below.')

from urllib.request import urlopen
with urlopen(f'http://127.0.0.1:{PORT}/', timeout=15) as response:
    page_status = response.status
    page_preview = response.read(300).decode('utf-8', errors='replace')
if page_status != 200 or '<html' not in page_preview.lower():
    raise RuntimeError(f'Unexpected home-page response. HTTP {page_status}: {page_preview}')

from google.colab.output import eval_js
url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
html = (
    '<p><a href="' + url + '" target="_blank" '
    'style="display:inline-block;padding:13px 19px;background:#5f52e8;color:#fff;'
    'border-radius:8px;text-decoration:none;font-weight:700">'
    'Open SoftMeta Audio and Avatar Studio</a></p>'
    '<p style="font-size:13px;color:#667085">Generate Video uses stable Ditto PyTorch mode. '
    'Test 10-20 seconds before a 10-30 minute checkpointed render.</p>'
)
display(HTML(html))
print('Home page check: HTTP 200 OK')
print('Server PID:', process.pid)
print('Server log:', LOG)


## Recent server log


In [ ]:
from pathlib import Path
log = Path('/content/softmeta_chatterbox_v092.log')
print(log.read_text(errors='replace')[-25000:] if log.exists() else 'No server log yet.')


## Optional: Stop the server

Leave `STOP_SERVER` disabled during normal use. Enable it only when you intentionally
want to stop the web server.


In [ ]:
STOP_SERVER = False  # @param {type:"boolean"}

import os
import signal
import subprocess
from pathlib import Path

pid_file = Path('/content/softmeta_chatterbox_v092.pid')
if not STOP_SERVER:
    print('Server remains running. Set STOP_SERVER to True only when you want to stop it.')
else:
    if pid_file.exists():
        try:
            os.kill(int(pid_file.read_text().strip()), signal.SIGTERM)
        except Exception:
            pass
        pid_file.unlink(missing_ok=True)
    subprocess.run("lsof -t -i:8004 | xargs -r kill -9", shell=True, check=False)
    print('Server stopped.')
